# RT-DETR-L — GenGUI / gengui13
Thesis experiment: `rtdetr_l_gengui_gengui13`

**Datasets required (add via right panel):** `uidet-gengui13`, `uidet-code`

**Secret required:** `WANDB_API_KEY`

In [ ]:
!pip install -q pycocotools wandb pyyaml ultralytics

In [ ]:
import os, sys, glob
from pathlib import Path

WORKING = Path('/kaggle/working')

def find_input(slug):
    for pattern in [f'/kaggle/input/{slug}',
                    f'/kaggle/input/datasets/*/{slug}',
                    f'/kaggle/input/datasets/*/*/{slug}']:
        matches = glob.glob(pattern)
        if matches:
            return Path(matches[0])
    raise FileNotFoundError(f"Dataset '{slug}' not found under /kaggle/input/")

DATA_INPUT = find_input('uidet-gengui13')
CODE_INPUT = find_input('uidet-code')

print('DATA_INPUT:', DATA_INPUT)
print('CODE_INPUT:', CODE_INPUT)

sys.path.insert(0, str(CODE_INPUT / 'src'))

import uidet
print('uidet imported OK:', uidet.__file__)


In [ ]:
import yaml, shutil

actual_images      = DATA_INPUT / 'images'
actual_labels      = DATA_INPUT / 'labels'
actual_annotations = DATA_INPUT / 'annotations'

print('images exists:     ', actual_images.exists())
print('labels exists:     ', actual_labels.exists())
print('annotations exists:', actual_annotations.exists())

work_data = WORKING / 'gengui13'
work_data.mkdir(parents=True, exist_ok=True)

# Symlink images and labels (large)
for name, src in [('images', actual_images), ('labels', actual_labels)]:
    link = work_data / name
    if link.is_symlink(): link.unlink()
    link.symlink_to(src)
    print(f'{name} link ok:', link.exists())

# Copy annotation JSONs (needed for COCO eval)
ann_dir = work_data / 'annotations'
if ann_dir.is_symlink(): ann_dir.unlink()
ann_dir.mkdir(exist_ok=True)
for f in actual_annotations.glob('*.json'):
    shutil.copy2(f, ann_dir / f.name)
    print('copied:', f.name)

# Write corrected data.yaml
with open(DATA_INPUT / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = str(work_data)
data_yaml_path = work_data / 'data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print('data.yaml written OK')
print(data_cfg)

In [ ]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
    print('WandB logged in via Kaggle secret')
except Exception as e:
    print(f'Secret not found ({e}), falling back...')
    wandb.login()

In [ ]:
from uidet.models.base import TrainConfig, build_detector
from uidet.train import get_class_names

EXP_NAME   = 'rtdetr_l_gengui_gengui13'
MODEL_NAME = 'rtdetr_l'
DATASET    = 'gengui'
TAXONOMY   = 'gengui13'

out_dir = WORKING / 'results_v2' / EXP_NAME
out_dir.mkdir(parents=True, exist_ok=True)

class_names = get_class_names(data_yaml_path)
print(f'{len(class_names)} classes: {class_names}')

cfg = TrainConfig(
    name=EXP_NAME,
    out_dir=out_dir,
    data_yaml=data_yaml_path,
    coco_val_json=ann_dir / f'{DATASET}_{TAXONOMY}_val.json',
    coco_test_json=ann_dir / f'{DATASET}_{TAXONOMY}_test.json',
    epochs=120,
    batch=8,
    imgsz=640,
    lr0=0.0001,
    seed=42,
    device='0',
    workers=4,
    amp=True,
    extra={
        'patience': 30,
        'cos_lr': True,
    },
    wandb_project='uidet-thesis',
    wandb_entity=None,
    wandb_tags=['kaggle', 'rtdetr', DATASET, TAXONOMY],
)
print('Config OK ->', out_dir)

In [ ]:
import json
from uidet.train import _wandb_init, _wandb_log_epochs_from_csv, _wandb_log_summary, _wandb_finish

cfg_dict = {
    'name': EXP_NAME, 'model': MODEL_NAME,
    'dataset': DATASET, 'taxonomy': TAXONOMY,
    'wandb_project': 'uidet-thesis', 'wandb_entity': None, 'wandb_tags': ['kaggle'],
}
wandb_run = _wandb_init(cfg_dict, cfg, class_names)

detector = build_detector(MODEL_NAME, num_classes=len(class_names), class_names=class_names)
print(f'Training {MODEL_NAME} on {DATASET}/{TAXONOMY} ({len(class_names)} classes)...')

weights = detector.train(cfg)
print('Best weights:', weights)

# Log per-epoch curves from results.csv (Ultralytics writes this during training)
_wandb_log_epochs_from_csv(out_dir, wandb_run)

In [ ]:
import time
from uidet.utils.metrics import coco_evaluate, detections_to_coco, format_metrics_table

detector.load(weights)

gt_path = ann_dir / f'{DATASET}_{TAXONOMY}_test.json'
gt      = json.loads(gt_path.read_text())
items   = [(im['id'], work_data / im['file_name']) for im in gt['images']]
name_to_cat_id = {c: i + 1 for i, c in enumerate(class_names)}

predictions, t0 = [], time.perf_counter()
for i in range(0, len(items), 4):
    chunk = items[i:i+4]
    batch_dets = detector.predict_batch([p for _, p in chunk], conf=0.001, iou=0.6)
    for image_id, dets in zip([iid for iid, _ in chunk], batch_dets):
        predictions.extend(detections_to_coco(dets, image_id, name_to_cat_id))
dt = time.perf_counter() - t0

eval_dir = out_dir / 'coco_eval_test'
metrics  = coco_evaluate(str(gt_path), predictions, eval_dir)
metrics['inference_seconds'] = dt
metrics['inference_fps']     = len(items) / dt
(eval_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(format_metrics_table(metrics))

In [ ]:
if wandb_run is not None:
    wandb.log({
        'test/mAP':   metrics['mAP'],
        'test/mAP50': metrics['mAP50'],
        'test/mAP75': metrics['mAP75'],
        **{f'test/AP/{cls}':   v['AP']   for cls, v in metrics['per_class'].items()},
        **{f'test/AP50/{cls}': v['AP50'] for cls, v in metrics['per_class'].items()},
    })
    print('COCO metrics logged to WandB')

In [ ]:
_wandb_log_summary(wandb_run, metrics, split='test')
_wandb_finish(wandb_run)

import shutil
shutil.make_archive(str(WORKING / EXP_NAME), 'zip', str(out_dir))
print('Download:', str(WORKING / EXP_NAME) + '.zip')
print(json.dumps(metrics, indent=2))